# 2. Your first PySpark DataFrames

The code of this module's lessons in [Learn Apache Spark by Doing](https://dataenglab.com/courses/learn-apache-spark-by-doing/), in lesson order, on Apache Spark 4.2.0 in local mode: one machine, no cluster.

Colab runs this notebook in your own Google account: sign in with Google to run the cells, and use *File → Save a copy in Drive* to keep your changes. Nothing runs on DataEngLab and nothing is installed on your computer.

Run the cells from top to bottom. Under a cell, the result the lesson shows is there to compare with yours.

Not available in Colab: the Spark UI (its port is not reachable from a notebook). Where a lesson sends you to it, run that part on your machine; here, `df.explain()` prints the plan.

In [ ]:
# Setup: Java 17+, PySpark 4.2.0 and the webshop data. About a minute the first time.
import os, re, subprocess, sys, urllib.request, zipfile

PYSPARK_VERSION = "4.2.0"
DATA_URL = 'https://github.com/dataeng-lab/learn-apache-spark-by-doing/raw/v2/datasets.zip'

if "google.colab" in sys.modules:
    version = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    major = re.search(r'version "(?:1\.)?(\d+)', version)
    if not major or int(major.group(1)) < 17:       # PySpark 4 does not start on Java 8 or 11
        subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                       shell=True, check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"], check=True)

if not os.path.isdir("datasets"):
    if DATA_URL is None:
        raise RuntimeError("datasets/ is missing: upload the course's datasets folder in the Files panel")
    urllib.request.urlretrieve(DATA_URL, "datasets.zip")
    zipfile.ZipFile("datasets.zip").extractall(".")

print("datasets:", sorted(os.listdir("datasets")))

## Set up PySpark and load the webshop data

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/set-up-pyspark-and-the-webshop-data/)

The lesson saves this as `check_spark.py`.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("webshop-setup").master("local[*]").getOrCreate()
print(spark.version)

The lesson saves this as `tour_webshop.py`.

In [ ]:
orders = spark.read.csv("datasets/orders.csv", header=True, inferSchema=True)
customers = spark.read.csv("datasets/customers.csv", header=True, inferSchema=True)

print("orders rows    :", orders.count())
print("customers rows :", customers.count())
print("distinct orders:", orders.select("order_id").distinct().count())
orders.printSchema()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
orders rows    : 4235
customers rows : 500
distinct orders: 4000

--- orders.printSchema() (inferSchema=True) ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
```

## SparkSession: your one entry point

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/sparksession-and-reading-data/)

The lesson saves this as `session.py`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("webshop-ingest")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [ ]:
first = SparkSession.builder.appName("job-a").getOrCreate()
second = SparkSession.builder.appName("job-b").getOrCreate()
# second is first; the app is still named job-a

Not run here: its last line reads warehouse/orders/, which the next cell writes: run it after the next cell.

```python
orders = spark.read.csv("datasets/orders.csv", header=True, inferSchema=True)
events = spark.read.json("datasets/order_events/")
curated = spark.read.parquet("warehouse/orders/")
```

The lesson saves this as `write_curated.py`.

In [ ]:
(
    orders
    .write
    .mode("overwrite")
    .parquet("warehouse/orders/")
)

## Schemas: inference is a convenience, not a contract

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/schemas-inference-vs-explicit/)

The lesson saves this as `revenue_by_status.py`.

In [ ]:
# This cell fails on purpose: the lesson explains why. try/except only keeps "Run all" going.
try:
    from pyspark.sql import functions as F

    orders = spark.read.csv("datasets/orders.csv", header=True, inferSchema=True)
    revenue = orders.groupBy("status").agg(F.sum("amount").alias("revenue"))
    revenue.show()
except Exception as error:
    print(type(error).__name__ + ":", str(error).splitlines()[0])

The lesson saves this as `orders_schema.py`.

In [ ]:
from pyspark.sql.types import (
    DecimalType, IntegerType, StringType, StructField, StructType, TimestampType,
)

ORDERS_SCHEMA = StructType([
    StructField("order_id", IntegerType(), nullable=False),
    StructField("customer_id", IntegerType(), nullable=True),
    StructField("order_ts", TimestampType(), nullable=True),
    StructField("status", StringType(), nullable=True),
    StructField("amount", DecimalType(10, 2), nullable=True),
    StructField("currency", StringType(), nullable=True),
    StructField("updated_at", TimestampType(), nullable=True),
])

orders = spark.read.csv("datasets/orders.csv", header=True, schema=ORDERS_SCHEMA)

The lesson saves this as `quarantine.py`.

In [ ]:
schema_with_bad = ORDERS_SCHEMA.add("_corrupt", StringType(), nullable=True)

orders = (
    spark.read
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt")
    .schema(schema_with_bad)
    .csv("datasets/orders.csv", header=True)
)

## Select, filter and column expressions

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/select-filter-and-expressions/)

In [ ]:
from pyspark.sql import functions as F

F.col("amount") * 2

The lesson saves this as `select_forms.py`.

In [ ]:
orders.select("order_id", "status", "amount")
orders.select(F.col("order_id"), F.col("status"), F.col("amount"))

The lesson saves this as `filter_orders.py`.

In [ ]:
delivered = orders.filter(F.col("status") == "delivered")
large = orders.filter((F.col("amount") >= 500) & (F.col("status") != "cancelled"))

In [ ]:
orders.filter(F.col("status") == "delivered")   # result thrown away
print(orders.count())                            # still 4235

The lesson saves this as `pipeline.py`.

In [ ]:
large_delivered = (
    orders
    .filter(F.col("status") == "delivered")
    .filter(F.col("amount") >= 500)
    .select(
        F.col("order_id"),
        F.col("customer_id"),
        F.col("amount").alias("order_value"),
    )
)

The lesson saves this as `derive.py`.

In [ ]:
enriched = (
    orders
    .withColumn("amount_eur", F.col("amount").cast("decimal(12,2)"))
    .withColumn("is_cancelled", F.col("status") == "cancelled")
)

The lesson saves this as `antipattern.py`.

In [ ]:
for column in ["order_id", "customer_id", "amount"]:
    orders = orders.withColumn(column, F.col(column).cast("string"))

The lesson saves this as `one_projection.py`.

In [ ]:
cast_to_string = ["order_id", "customer_id", "amount"]

typed = orders.select(*[
    F.col(c).cast("string").alias(c) if c in cast_to_string else F.col(c)
    for c in orders.columns
])

## Types, nulls and aggregations

[Read the lesson](https://dataenglab.com/courses/learn-apache-spark-by-doing/lessons/types-nulls-and-aggregations/)

**Starting point for the next cell.** Not part of the lesson text.

In [ ]:
# The previous lesson ended by casting columns to strings. Start again from the explicit schema.
orders = spark.read.csv("datasets/orders.csv", header=True, schema=ORDERS_SCHEMA)

The lesson saves this as `null_logic.py`.

In [ ]:
from pyspark.sql import functions as F

big = orders.filter(F.col("amount") >= 500)
small = orders.filter(F.col("amount") < 500)
# big.count() + small.count() is not orders.count()

The lesson saves this as `null_handling.py`.

In [ ]:
orders.filter(F.col("customer_id").isNull())
orders.filter(F.col("customer_id").isNotNull())

orders.fillna({"amount": 0})              # replace NULLs in named columns
orders.dropna(subset=["customer_id"])     # drop rows where customer_id is NULL
orders.na.drop(how="all")                 # drop rows where every column is NULL

The lesson saves this as `counting.py`.

In [ ]:
orders.selectExpr("count(*) AS count_star", "count(customer_id) AS count_col").show()

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
rows                       : 4235
rows with NULL customer_id : 83
share                      : 1.96%

count(customer_id) ignores NULLs, count(*) does not:
+----------+---------+
|count_star|count_col|
+----------+---------+
|      4235|     4152|
+----------+---------+
```

The lesson saves this as `revenue_by_status.py`.

In [ ]:
from pyspark.sql import functions as F

(
    orders                                     # read with the explicit DECIMAL schema
    .groupBy("status")
    .agg(
        F.count("*").alias("orders"),
        F.sum("amount").alias("revenue"),
    )
    .orderBy(F.col("orders").desc())
    .show(truncate=False)
)

What the lesson shows. It comes from a longer version of this example, so the numbers match but your cell may print less:

```text
--- orders per status (explicit schema, malformed amounts are NULL) ---
+---------+------+----------+
|status   |orders|revenue   |
+---------+------+----------+
|created  |4000  |2526490.94|
|cancelled|68    |35865.75  |
|delivered|62    |34586.18  |
|paid     |55    |34786.04  |
|shipped  |50    |27446.09  |
+---------+------+----------+
```